**SSD**

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

url = 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image2.jpg'
path = tf.keras.utils.get_file('ssd_beach.jpg', origin=url)
image = np.array(Image.open(path).convert('RGB'))

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.axis('off')
plt.show()


In [ ]:
import tensorflow as tf
****************
import matplotlib.patches as patches

ssd_model = hub.load('https://tfhub.dev/tensorflow/ssd_mobilenet_v2/fpnlite_320x320/1')

In [ ]:
import requests, re

url = "https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/data/mscoco_label_map.pbtxt"
****************
coco_names = {int(i): name for i, name in re.findall(r'id:\s*(\d+)\s+display_name:\s*"([^"]+)"', text)}

print(coco_names)

In [ ]:
predictions = ssd_model(tf.convert_to_tensor(image[None, ...], dtype=tf.uint8))

scores = predictions['detection_scores'][0].numpy()
****************

for i in np.argsort(scores)[-10:][::-1]:
    print(f"{coco_names.get(classes[i], classes[i])}: {scores[i]:.3f}")

In [ ]:
****************
    boxes = predictions['detection_boxes'][0].numpy()
    scores = predictions['detection_scores'][0].numpy()
    classes = predictions['detection_classes'][0].numpy().astype(int)
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(image)
    count = 0

    for box, score, label in zip(boxes, scores, classes):
        if score < threshold: continue
        top, left, bottom, right = box
        x, y = left * width, top * height
        w, h = (right - left) * width, (bottom - top) * height
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        ax.text(x, y, f'{coco_names.get(label, label)}: {score:.2f}', color='black', bbox=dict(facecolor='lime', alpha=0.8))
        count += 1

    ax.axis('off')
    plt.show()
****************

show_ssd(image, predictions, threshold=0.2)

In [ ]:
****************


**YOLO**

In [ ]:
!pip -q install ultralytics

In [ ]:
****************
from ultralytics.data.utils import check_det_dataset
from IPython.display import display
from PIL import Image
from pathlib import Path
****************


In [ ]:
****************
print("First five pretrained classes:", list(model.names.values())[:5])
print("Pill is a COCO class:", "pill" in model.names.values())

In [ ]:
****************
val_images = sorted(Path(dataset["val"]).glob("*.jpg")) + sorted(Path(dataset["val"]).glob("*.png"))

sample = val_images[0]
print("Class names:", dataset["names"])
print("Validation image:", sample)
display(Image.open(sample).resize((500, 400)))

In [ ]:
****************
display(Image.fromarray(before.plot()[..., ::-1]).resize((600, 400)))
print("Detected classes:", [model.names[int(c)] for c in before.boxes.cls.tolist()])

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
model.train(data="medical-pills.yaml", epochs=12, imgsz=416, batch=8, device=device, project="runs_yolo_lab", name="pills", exist_ok=True)
****************
print("Best checkpoint:", best_path)

In [ ]:
metrics = model.val(data="medical-pills.yaml", imgsz=416, device=device)
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
****************
print(f"mAP50-95:  {metrics.box.map:.3f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("runs/detect/runs_yolo_lab/pills/results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, loss in zip(axes, ["box_loss", "cls_loss", "dfl_loss"]):
****************
    ax.plot(df["epoch"], df[f"val/{loss}"], label="Validation", marker="o")
    ax.set_title(loss)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
****************
after = fine_tuned.predict(source=str(sample), conf=0.15, verbose=False)[0]
display(Image.fromarray(after.plot()[..., ::-1]).resize((600, 400)))
print("Pills detected:", len(after.boxes))

In [ ]:
result = fine_tuned.predict(source=str(sample), conf=0.35, iou=0.5, verbose=False)[0]
display(Image.fromarray(result.plot()[..., ::-1]).resize((600, 400)))
print("Pills shown:", len(result.boxes))